# Session 3.2 — Feedback Loops & Eval-Driven Development

## What We're Building Today

Today's session is **SHIPPED** — the feedback infrastructure is pre-built and working in `app/feedback.py`.
You are wiring it into your chat UI and learning the feedback-to-eval pipeline.

By the end of this session you will have:
- **Feedback capture** working in your Streamlit app (thumbs up/down on every response)
- **Phoenix annotations** — feedback stored directly on traces, not in separate files
- **At least one negative trace** converted into a golden test case
- **An expanded evaluation run** that includes feedback-derived test cases

## Learning Objectives

1. **Capture** explicit user feedback (thumbs up/down) using `st.feedback()`
2. **Store** feedback as Phoenix span annotations with full trace context
3. **Convert** negative feedback into golden test cases that expand the evaluation framework
4. **Run** the expanded golden set and measure improvement or regression
5. **Describe** the eval-driven development cycle: feedback → test case → fix → re-evaluate → ship

## Session Theme

> "Every thumbs down is a test case you did not have."

In [ ]:
# Path setup — run this cell FIRST
# Notebooks run from their subdirectory but pipeline modules are at the project root
import sys
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parent.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

# Load environment variables from project root
from dotenv import load_dotenv
load_dotenv(_PROJECT_ROOT / ".env")

print(f"Project root: {_PROJECT_ROOT}")
print(f"Path configured: pipeline imports will work")

In [ ]:
# Load mermaid diagram support — run once per notebook
%load_ext mermaid_magic

In [ ]:
# Import verification — pipeline modules + feedback infrastructure
from pipeline.retrieval.naive import naive_retrieve
from pipeline.generation.generate import call_claude
from pipeline.context.assembler import contextualize_query, assemble_context
from pipeline.context.manager import manage_history
from pipeline.safety.guard import validate_input, build_hardened_prompt, validate_output
from app.feedback import submit_feedback, get_feedback_summary

print("All pipeline imports successful!")
print("  - pipeline.retrieval.naive")
print("  - pipeline.generation.generate")
print("  - pipeline.context.assembler")
print("  - pipeline.context.manager")
print("  - pipeline.safety.guard")
print("  - app.feedback (submit_feedback, get_feedback_summary)")

In [ ]:
# Phoenix connection verification
from phoenix.client import Client

px_client = Client()
print(f"Phoenix client connected")
print(f"Ready to read span annotations and datasets")

In [ ]:
# ChromaDB + Claude API verification
from pipeline.ingestion.store import get_collection

collection = get_collection()
count = collection.count()
print(f"ChromaDB collection: {collection.name}")
print(f"Documents loaded: {count} chunks")
assert count > 0, "No chunks found! Did you run ingestion?"

# Claude API test
response = call_claude("Say hello in 3 words")
print(f"Claude says: {response}")

### Setup Checklist

- [ ] All pipeline imports successful (including `app.feedback`)
- [ ] Phoenix client connected
- [ ] ChromaDB collection has chunks loaded
- [ ] Claude API responds

If anything above failed, check your `.env` file and confirm Phoenix is running.

## Phoenix Tracing — Code Changes Required

Before feedback can work, your app needs **Phoenix tracing**. You need to make changes to **three files**. Follow each section exactly.

---

### File 1: `app/feedback.py` — Fix the API path

Open `app/feedback.py`. Find this line (around line 56):

```python
client.annotations.add_span_annotation(
```

**Change it to:**

```python
client.spans.add_span_annotation(
```

Also find this line (around line 82):

```python
spans_df = client.get_spans_dataframe()
```

**Change it to:**

```python
spans_df = client.spans.get_spans_dataframe()
```

> **Why this matters:** The original code calls a path that doesn't exist (`client.annotations`). It fails silently because of `except Exception: pass`. This is a real-world lesson — broad exception handling can hide bugs for weeks.

---

### File 2: `app/rag.py` — Add pipeline tracing spans

These changes wrap your pipeline in spans so Phoenix can see retrieval, generation, and the full pipeline.

**Step 2a:** Add these imports at the top of `rag.py` (after the existing `from opentelemetry import trace`):

```python
from opentelemetry.trace import StatusCode

_tracer = trace.get_tracer("rag-pipeline")
```

**Step 2b:** Inside `get_response()`, replace the existing span capture code:

```python
# OLD — remove this:
span = trace.get_current_span()
ctx = span.get_span_context()
span_id = format(ctx.span_id, '016x') if ctx.span_id else ""
```

**With this — wrap the ENTIRE function body:**

```python
    with _tracer.start_as_current_span(
        "rag_pipeline",
        attributes={"input.value": question, "openinference.span.kind": "CHAIN"},
    ) as pipeline_span:
        ctx = pipeline_span.get_span_context()
        span_id = format(ctx.span_id, '016x') if ctx.span_id else ""

        # ... rest of your pipeline code, indented one level ...
```

**Step 2c:** Wrap the `retrieve()` call in a RETRIEVER span:

```python
        with _tracer.start_as_current_span("retrieve_chunks") as ret_span:
            ret_span.set_attribute("openinference.span.kind", "RETRIEVER")
            ret_span.set_attribute("input.value", rewritten)
            sources = retrieve(rewritten, top_k=5)
            ret_span.set_attribute("retrieve.top_k", 5)
            ret_span.set_attribute("retrieve.n_results", len(sources))
            if sources:
                ret_span.set_attribute("retrieve.sources",
                    ", ".join(s.get("metadata", {}).get("source", "") for s in sources))
                ret_span.set_attribute("retrieve.top_score", float(sources[0].get("score", 0)))
            ret_span.set_status(StatusCode.OK)
```

**Step 2d:** Wrap `call_claude()` in an LLM span:

```python
        with _tracer.start_as_current_span("generate_answer") as gen_span:
            gen_span.set_attribute("openinference.span.kind", "LLM")
            gen_span.set_attribute("input.value", user_message)
            answer = call_claude(user_message, system_prompt=system_prompt, temperature=0.0)
            gen_span.set_attribute("output.value", answer[:500])
            gen_span.set_status(StatusCode.OK)
```

**Result in Phoenix:**
```
rag_pipeline (CHAIN)
├── retrieve_chunks (RETRIEVER) — sources, scores, top_k
├── generate_answer (LLM) — question and answer
└── Claude API call (auto-instrumented)
```

---

### File 3: `app/main.py` — Initialize tracing + sessions

**Step 3a:** Add imports at the top (after existing imports):

```python
import os
import uuid
from dotenv import load_dotenv

load_dotenv(_PROJECT_ROOT / ".env")
```

**Step 3b:** Add Phoenix initialization (after imports, before branding):

```python
if "phoenix_initialized" not in st.session_state:
    try:
        from phoenix.otel import register
        register(
            project_name=os.getenv("PHOENIX_PROJECT_NAME", "ai3"),
            auto_instrument=True,
        )
    except Exception:
        pass
    st.session_state.phoenix_initialized = True

if "session_id" not in st.session_state:
    st.session_state.session_id = str(uuid.uuid4())
```

**Step 3c:** In your Step 5 (chat handler), wrap `get_response()`:

```python
from openinference.instrumentation import using_attributes

with using_attributes(
    session_id=st.session_state.session_id,
    user_id="student",
    tags=["streamlit"],
):
    response = get_response(prompt, st.session_state.messages)
```

---

After making all three changes, **restart Streamlit** and ask a question. Open Phoenix — you should see traces with the full pipeline hierarchy.

---

## Developer View vs User View

Right now, the only way you know if your app is good is the **golden test set** from Session 1.2.
That is 10 queries that *you* — the developer — wrote. You thought of the questions.
You defined the expected answers. That is the **developer's view** of quality.

Users think differently. They ask questions you never imagined. They phrase things in ways
your test set does not cover. They expect information you did not know was in the corpus.

```
DEVELOPER VIEW              USER VIEW
+--------------------+      +--------------------+
| Golden test set    |      | Real questions     |
| 10 curated queries |      | Unexpected phrasing|
| Known good answers |      | Edge cases         |
| Controlled         |      | Messy, diverse     |
+--------------------+      +--------------------+
        |                           |
   Necessary                   Also necessary
   (baseline)                  (reality check)
```

The golden test set is your **baseline** — it prevents regressions.
User feedback is your **reality check** — it tells you what the golden set is missing.
You need both.

## Three Types of Feedback

| Type | Example | Coverage | Signal Quality | Effort to Capture |
|------|---------|----------|----------------|-------------------|
| **Explicit** | Thumbs up/down, comments | 5-10% of interactions | High — direct user judgment | Low — UI widget |
| **Implicit** | Query reformulation, abandonment | 100% of interactions | Medium — requires interpretation | Medium — log analysis |
| **System** | Retrieval scores, latency, token usage | 100% of interactions | Low — correlation, not causation | Low — already in pipeline |

**Today we wire explicit feedback** — thumbs up/down — into your app.
The infrastructure is already built in `app/feedback.py`.
You add about 5 lines to your chat display, and feedback flows into Phoenix as span annotations.

We already have retrieval scores, source information, and timing from the pipeline traces.
The feedback annotation completes the picture.

## Why Phoenix-Native?

Feedback is stored as **span annotations** in Phoenix — not in a local JSON file.

Why? Because the annotation lives on the **same trace** that has the query, the response,
the retrieval scores, and the latency. Everything in one place.

Compare that to a bare "thumbs down" in a separate file with no context.
You know something was bad but not *what* or *why*.
Phoenix-native feedback gives you the full picture without any extra logging.

When you open Phoenix and find a negative annotation, you see:
- The exact query the user asked
- The response the user rated
- Which chunks were retrieved (and their scores)
- How long the whole thing took
- The user's rating — right there on the span

---

## Study: `app/feedback.py`

This module was shipped as complete, working code at the Session 3.1 freeze.
Two key functions:

- `submit_feedback(span_id, score)` — creates a Phoenix annotation on a span
- `get_feedback_summary()` — queries Phoenix for annotation counts and recent negatives

Read through both functions. Understand what they do before you wire them in.

In [ ]:
# Study submit_feedback() — how feedback gets stored
import inspect
from app.feedback import submit_feedback

print(inspect.getsource(submit_feedback))

In [ ]:
# Study get_feedback_summary() — how feedback is retrieved
import inspect
from app.feedback import get_feedback_summary

print(inspect.getsource(get_feedback_summary))

**Key observations from studying `feedback.py`:**

- `submit_feedback()` attaches a rating directly to a Phoenix span as an annotation
- The annotation includes label ("positive"/"negative"), score (1.0/0.0), and explanation
- `_PHOENIX_AVAILABLE` flag ensures the app works without Phoenix (graceful degradation)
- `get_feedback_summary()` reads annotations back from Phoenix

> **You should have already fixed the API paths in File 1 above.** If not, go back and do that now — without the fix, feedback silently fails.

---

## Wire Feedback Into Your UI

Open `app/main.py`. You need to add three things:

### A. Import feedback functions (add after other imports)

```python
from app.feedback import submit_feedback, get_feedback_summary
```

### B. Add feedback callbacks and render function (add before the display loop)

```python
def _save_feedback(index):
    feedback_value = st.session_state[f"fb_{index}"]
    st.session_state.messages[index]["feedback"] = feedback_value
    span_id = st.session_state.messages[index].get("span_id", "")
    if span_id:
        submit_feedback(span_id, feedback_value)
    st.session_state.conversations[st.session_state.current_chat] = (
        st.session_state.messages.copy()
    )
    if feedback_value == 1:
        st.toast("Thanks for the positive feedback!")
    else:
        st.toast("Thanks — you can add details below.")


def _save_feedback_note(index):
    note = st.session_state.get(f"note_{index}", "")
    if not note:
        return
    span_id = st.session_state.messages[index].get("span_id", "")
    if span_id:
        submit_feedback(span_id, 0, note=note)
    st.session_state.messages[index]["feedback_note"] = note
    st.session_state.conversations[st.session_state.current_chat] = (
        st.session_state.messages.copy()
    )
    st.toast("Detailed feedback submitted!")


def render_feedback(index):
    message = st.session_state.messages[index]
    existing_fb = message.get("feedback", None)
    st.session_state[f"fb_{index}"] = existing_fb
    st.feedback(
        "thumbs",
        key=f"fb_{index}",
        disabled=existing_fb is not None,
        on_change=_save_feedback,
        args=[index],
    )
    if existing_fb == 0 and not message.get("feedback_note"):
        st.text_input(
            "What went wrong?",
            key=f"note_{index}",
            placeholder="Help us improve (press Enter to submit)",
            on_change=_save_feedback_note,
            args=[index],
        )
    elif message.get("feedback_note"):
        st.caption(f"Your note: _{message['feedback_note']}_")
```

### C. Call `render_feedback()` in the display loop AND Step 5

In the message display loop (Step 3):
```python
for i, message in enumerate(st.session_state.get("messages", [])):
    with st.chat_message(message["role"]):
        st.markdown(message["content"])
        if message["role"] == "assistant":
            render_feedback(i)
```

In Step 5 (after displaying the response), store `span_id` and render:
```python
    # Append BEFORE rendering so callback can find the message
    st.session_state.messages.append({
        "role": "assistant",
        "content": response.answer,
        "span_id": response.span_id,
    })
    # ... then in the chat_message block:
    render_feedback(len(st.session_state.messages) - 1)
```

### Design Decisions

**Why `span_id`?** Every time your app answers a question, Phoenix creates a trace with a span ID.
The span ID links the feedback annotation back to the specific interaction — query, response, sources, scores.
Without it, feedback has no context.

**Why thumbs, not stars?** Thumbs up/down is a single bit of information: good or bad.
That is enough to identify failures. Star ratings add complexity without much signal —
is a 3-star response "okay" or "bad"? With thumbs, there is no ambiguity.

**Where does the widget go?** Inside the chat bubble, right below the response text.
Users see the answer and immediately have the option to rate it. No extra clicks, no separate page.

In [ ]:
# Verify the feedback module imports work — this confirms your app can use it
from app.feedback import submit_feedback, get_feedback_summary

print("submit_feedback() is ready")
print(f"  Parameters: span_id (str), feedback_value (int), note (str, optional)")
print()
print("get_feedback_summary() is ready")
print(f"  Returns: dict with keys: total, positive, negative, recent_negative")

---

## Test Feedback

Now that you have wired `st.feedback()` into `app/main.py`:

1. **Run your app** (`streamlit run app/main.py` from the project root)
2. **Ask a question** — something you know the app handles well
3. **Give it thumbs up**
4. **Ask another question** — something tricky or edge-case
5. **Give it thumbs down**
6. Come back here and run the cell below to check feedback was recorded

In [ ]:
# Check feedback summary from Phoenix
from app.feedback import get_feedback_summary

summary = get_feedback_summary()

print("Feedback Summary")
print("=" * 40)
print(f"  Total feedback entries: {summary['total']}")
print(f"  Positive (thumbs up):   {summary['positive']}")
print(f"  Negative (thumbs down): {summary['negative']}")
print()

if summary["recent_negative"]:
    print("Recent Negative Feedback:")
    print("-" * 40)
    for entry in summary["recent_negative"]:
        query = entry.get("input", "(unknown)")[:80]
        print(f"  Query: {query}")
        answer = entry.get("output", "(unknown)")[:80]
        print(f"  Answer: {answer}...")
        print()
else:
    print("No negative feedback yet. Use your app and give some thumbs down!")

### Reflection

Look at the numbers above.

- If you have 0 feedback entries, go back to your app and give some ratings.
- If you have only positive feedback, try asking harder questions — edge cases, ambiguous queries, topics with thin corpus coverage.
- If you have negative feedback, you have data. That is the starting point for the next section.

---

## Review Negative Traces in Phoenix

This is where feedback becomes actionable.

**In the Phoenix UI:**
1. Navigate to your project's traces
2. Filter for traces with the `user-feedback` annotation
3. Find one with a negative label (score = 0)
4. Click into the trace and investigate:
   - What was the query?
   - What chunks were retrieved? What were the retrieval scores?
   - Was the right information in the chunks but the answer missed it? (Generation failure)
   - Or were the wrong chunks retrieved entirely? (Retrieval failure)

The trace tells you everything. This is the developer workflow:
user gives thumbs down → find the trace → diagnose the root cause.

In [ ]:
# Query Phoenix for spans with feedback annotations
from phoenix.client import Client

px = Client()

try:
    spans_df = px_client.spans.get_spans_dataframe()
    if spans_df is not None and not spans_df.empty:
        print(f"Total spans: {len(spans_df)}")
        print(f"Columns available: {list(spans_df.columns)[:10]}...")
        print()

        # Show recent spans with their input/output
        for idx, row in spans_df.head(5).iterrows():
            query = str(row.get("input.value", ""))[:60]
            output = str(row.get("output.value", ""))[:60]
            anns = row.get("annotations", [])
            feedback_label = ""
            if isinstance(anns, list):
                for ann in anns:
                    if isinstance(ann, dict) and ann.get("name") == "user-feedback":
                        feedback_label = f" [{ann.get('label', '?')}]"
            print(f"  Query: {query}")
            print(f"  Output: {output}...")
            print(f"  Feedback: {feedback_label or '(none)'}")
            print()
    else:
        print("No spans found. Use your app first to generate traces.")
except Exception as e:
    print(f"Could not query Phoenix: {e}")
    print("Make sure Phoenix is running and you have generated some traces.")

In [ ]:
%%mermaid
graph TD
    FB["User gives\nthumbs down"] --> LOG["Feedback logged\nwith full context"]
    LOG --> REV["Developer reviews\ntrace in Phoenix"]
    REV --> DIAG{"Root cause?"}
    DIAG -->|"Wrong chunks"| RET["Retrieval failure"]
    DIAG -->|"Right chunks,\nbad answer"| GEN["Generation failure"]
    DIAG -->|"Thin coverage"| CORPUS["Corpus gap"]
    RET --> TC["New golden\ntest case"]
    GEN --> TC
    CORPUS --> TC
    TC --> EVAL["Run expanded\ngolden set"]
    EVAL --> FIX["Make change\n(retrieval, prompt,\nchunking)"]
    FIX --> REEVAL["Re-run golden set\nVerify fix +\nno regression"]
    REEVAL --> SHIP["Ship"]

    style FB fill:#f8d7da,stroke:#721c24
    style TC fill:#fff3cd,stroke:#856404
    style EVAL fill:#cce5ff,stroke:#004085
    style REEVAL fill:#cce5ff,stroke:#004085
    style SHIP fill:#d4edda,stroke:#155724

### The Eval-Driven Development Cycle

This diagram is the core workflow:

1. **Capture** — User gives thumbs down. Feedback stored with full trace context.
2. **Review** — Developer finds the trace in Phoenix. Diagnoses root cause.
3. **Convert** — Turn the bad interaction into a golden test case.
4. **Test** — Run the expanded golden set. The new case fails (expected).
5. **Fix** — Change retrieval, prompt, chunking — whatever addresses the root cause.
6. **Re-test** — Run the golden set again. New case passes. Old cases still pass.
7. **Ship** — You have evidence — not vibes — that the fix works.

This is the same evaluation framework from Session 1.2. Same golden set format.
Same metrics. Same comparison methodology. The only difference: now the test cases
come from **real users**, not just your imagination.

---

## Convert Negative Feedback to Test Cases

Turn complaints into coverage.

The process:
1. Find a negative trace in Phoenix (or from `get_feedback_summary()`)
2. Read the query and the bad response
3. Determine what the **correct** answer should be (check the corpus)
4. Format it as a golden test case entry
5. Add it to your golden set

**Not every thumbs down becomes a test case.** You — the developer — decide what is worth testing.
If the query was genuinely ambiguous, maybe the answer was fine and the user was wrong.
If the query was clear and the answer was clearly bad, that is a test case.

In [ ]:
# Extract negative feedback traces for conversion
from app.feedback import get_feedback_summary

summary = get_feedback_summary()
negative_traces = summary.get("recent_negative", [])

print(f"Found {len(negative_traces)} recent negative feedback entries")
print()

for i, trace in enumerate(negative_traces, 1):
    print(f"--- Negative Entry #{i} ---")
    print(f"  Query:     {trace.get('input', '(unknown)')[:100]}")
    print(f"  Response:  {trace.get('output', '(unknown)')[:100]}...")
    print(f"  Timestamp: {trace.get('timestamp', '(unknown)')}")
    print()

In [ ]:
# Format as a golden test case entry
# This is the same schema as pipeline/eval/golden_set.py
#
# Fill in the values based on a negative trace you found above.
# Replace the placeholder strings with real data.

new_test_case = {
    "id": "feedback_001",
    "question": "PASTE THE QUERY FROM THE NEGATIVE TRACE",
    "expected_answer": (
        "WRITE THE CORRECT ANSWER HERE — check the corpus documents "
        "to determine what the answer SHOULD have been."
    ),
    "expected_source": ["source_document.md"],  # Which doc should be retrieved?
    "category": "feedback_derived",
    "difficulty": "medium",
    "history": [],  # empty for single-turn
}

print("New golden test case:")
print("=" * 50)
for key, value in new_test_case.items():
    display = str(value)[:80]
    print(f"  {key}: {display}")

### Human in the Loop

Notice the step you just did: **you** decided what the correct answer should be.
The user told you the answer was bad. You verified it by checking the corpus.
Then you wrote the expected answer.

This is human-in-the-loop evaluation. The system captures signal.
The developer interprets it and decides what to do.
No automated system can replace this judgment — not yet.

The value of the pipeline is that it makes the signal **actionable**:
- Without Phoenix: "Someone said the app was bad." (No context.)
- With Phoenix: "This query retrieved these chunks with these scores and produced this answer, and the user rated it negative." (Full diagnostic.)

---

## Run Expanded Evaluation

Before running the experiments, we sync the golden set to Phoenix (in case you added
feedback-derived test cases) and then run both correctness and safety experiments directly
through the Phoenix API.

You should see:
- **Original test cases still passing** — that is your regression check
- **New test cases from feedback may fail** — that is expected. Those failures are your improvement targets for Lab 2.

In [ ]:
# Sync golden set to Phoenix (adds any missing entries, including feedback-derived ones)
from phoenix.client import Client
from pipeline.eval.golden_set import GOLDEN_SET, get_dataset_name

px_client = Client()
dataset_name = get_dataset_name()

# Build rows from the full golden set
inputs = [{"question": e["question"], "history": e.get("history", [])} for e in GOLDEN_SET]
outputs = [{"expected_answer": e["expected_answer"], "expected_source": e["expected_source"]} for e in GOLDEN_SET]
metadata = [{"id": e["id"], "category": e["category"], "difficulty": e["difficulty"]} for e in GOLDEN_SET]

try:
    dataset = px_client.datasets.create_dataset(
        name=dataset_name,
        description="Northbrook golden test set (includes feedback-derived entries)",
        inputs=inputs,
        outputs=outputs,
        metadata=metadata,
    )
    print(f"Created fresh dataset: {dataset_name} ({len(GOLDEN_SET)} entries)")
except Exception as e:
    if "409" in str(e) or "already exists" in str(e).lower():
        print(f"Dataset '{dataset_name}' already exists.")
        dataset = px_client.datasets.get_dataset(dataset=dataset_name)
        existing = len(list(dataset.examples))
        print(f"  Current version has {existing} entries")
        if existing < len(GOLDEN_SET):
            print(f"  NOTE: You have fewer than {len(GOLDEN_SET)} entries. Consider re-creating the dataset.")
        else:
            print(f"  Full set is present.")
    else:
        raise

In [ ]:
# Run BOTH correctness and safety experiments via Phoenix API
from phoenix.client import Client
from pipeline.eval.golden_set import get_dataset_name
from pipeline.eval.adversarial_set import get_adversarial_dataset_name
from pipeline.eval.tasks import naive_task, safety_task
from pipeline.eval.evaluators import retrieval_hit, answer_addresses_question, safety_check

px_client = Client()
MODEL_NAME = "claude-sonnet-4-5"

# --- Correctness experiment (golden set) ---
golden_name = get_dataset_name()
print(f"Running correctness experiment on: {golden_name}")
golden_dataset = px_client.datasets.get_dataset(dataset=golden_name)

correctness_result = px_client.experiments.run_experiment(
    dataset=golden_dataset,
    task=naive_task,
    evaluators=[retrieval_hit, answer_addresses_question],
    experiment_name=f"post-feedback / {MODEL_NAME}",
    experiment_metadata={"pipeline": "with_feedback_cases", "model": MODEL_NAME},
)
print(f"  Correctness experiment done.")

# --- Safety experiment (adversarial set) ---
adversarial_name = get_adversarial_dataset_name()
print(f"\nRunning safety experiment on: {adversarial_name}")
adversarial_dataset = px_client.datasets.get_dataset(dataset=adversarial_name)

safety_result = px_client.experiments.run_experiment(
    dataset=adversarial_dataset,
    task=safety_task,
    evaluators=[safety_check],
    experiment_name=f"safety-post-feedback / {MODEL_NAME}",
    experiment_metadata={"pipeline": "safety", "model": MODEL_NAME},
)
print(f"  Safety experiment done.")

print(f"\n{'='*50}")
print(f"Both experiments complete. Check Phoenix:")
print(f"  Golden set -> {golden_name} -> Experiments tab")
print(f"  Adversarial set -> {adversarial_name} -> Experiments tab")

### Before/After Comparison

Open Phoenix and navigate to **Datasets → Experiments**.

Compare the latest experiment run against previous ones:
- Did the original 10 golden set queries still pass? (Regression check)
- Did any new feedback-derived test cases fail? (Expected — these are your Lab 2 targets)
- Did the safety experiment results change? (They should not, unless you modified guards)

Do not try to fix failures right now. The point of this step is to demonstrate the cycle:
feedback identifies the problem, test cases capture it, evaluation measures it.
**Lab 2 is where you do the fixing and measuring.**

---

## Lab 1 Collection + Lab 2 Check-In

### Lab 1 — Due Now

Lab 1 (GUI Customization) is due at the end of this session. Submit your repo link.

### Lab 2 — Progress Check

Lab 2 (Application Improvements) was assigned at Session 3.1. Due at the **start of Session 4.2**.

Quick check-in:
- Have you chosen a retrieval strategy? (HyDE, enriched, or both?)
- Have you started adversarial test cases in `pipeline/eval/student_attacks.py`?
- Have you run before/after experiments in Phoenix?

### Lab 2 Requirements Reminder

| # | Section | File | What You Can Change |
|---|---------|------|--------------------||
| 1 | Retrieval Strategy | `rag.py` import | `naive_retrieve` → `hyde_retrieve` or `enriched_retrieve` |
| 2 | System Prompt | `guard.py` | Persona, tone, grounding rules |
| 3 | History Management | `manager.py` | Keep-first-plus-last, summarization |
| 4 | Query Rewriting | `assembler.py` | Change prompt, adjust history window |
| 5 | Retrieval Parameters | `rag.py` | `top_k`, score thresholds |
| 6 | Context Assembly | `assembler.py` | Grouping, ordering, metadata format |
| 7 | Generation Settings | `rag.py` | Model, temperature, max_tokens |

**Customize at least 3 of these 7 sections.** Show before/after evidence in Phoenix.

---

## What You Built Today

- **Phoenix tracing** — `register()`, sessions, RETRIEVER/LLM/CHAIN spans
- **Feedback capture** — `st.feedback()` wired into your chat display, storing annotations on Phoenix traces
- **Feedback review** — queried Phoenix for negative annotations, inspected traces, diagnosed root causes
- **Test case conversion** — turned a negative feedback entry into a golden test case
- **Expanded evaluation** — ran the golden set and safety experiments with new test cases

### The Eval-Driven Development Cycle

```
Capture  →  Review  →  Convert  →  Test  →  Fix  →  Re-test  →  Ship
  |           |           |          |         |         |           |
  thumbs    Phoenix     golden     eval      change    eval       evidence
  down      trace       test case  (fail)    code      (pass)     not vibes
```

This cycle is how production AI systems get better. Not by guessing. Not by vibes.
By measuring. Every change is tested. Every improvement is proven with numbers.
Every regression is caught before it ships.

### Questions to Leave With

> **On Deployment:** "Your app works on your laptop. How do you get it running on someone else's machine?"

> **On Configuration:** "API keys, model settings, database paths — how do you manage all of that across environments?"

---

### Next Session: 4.1 — Deployment Readiness

- Docker containerization
- Configuration management across environments
- Getting your app running on someone else's machine

**Lab 1:** Collected today.
**Lab 2:** Keep building. Due at the start of Session 4.2.